# 02 – Giảm chiều: PCA và FAMD

Notebook chỉ gọi hàm trong `src/reduce.py` (issue #10 PCA, #11 FAMD). Dữ liệu: `clean` → `add_derived_features` → chia 7:3 có phân tầng, seed 0. Mọi mô hình giảm chiều **chỉ fit trên train**; validation chỉ transform. Hình lưu vào `results/figures/` (`pca_*.png`, `famd_*.png`). t-SNE và bảng so sánh PCA–FAMD thuộc issue #12, không làm ở đây.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.data import load_raw, TARGET
from src.preprocess import clean, add_derived_features, make_splits
from src import reduce as rd

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)
FIG = rd.FIG_DIR
K = 11  # số chiều giữ lại (1/3 của 33)

## 0. Dữ liệu và chia train / validation

In [2]:
df = add_derived_features(clean(load_raw()))
y = df[TARGET]
tr_idx, va_idx = make_splits(y, [0], {'7:3': 0.3})[('7:3', 0)]
df_tr, df_va = df.loc[tr_idx], df.loc[va_idx]   # index là nhãn dòng gốc -> dùng .loc
y_tr, y_va = y.loc[tr_idx], y.loc[va_idx]
pd.DataFrame({'số mẫu': [len(df_tr), len(df_va)], 'tỉ lệ vỡ nợ': [y_tr.mean(), y_va.mean()]},
             index=['train', 'validation']).round(4)

EDUCATION = 0 -> 4: 14 mẫu
EDUCATION = 5 -> 4: 280 mẫu
EDUCATION = 6 -> 4: 51 mẫu
MARRIAGE = 0 -> 3: 54 mẫu
Loại bản ghi trùng hoàn toàn: 35 dòng (còn 29965)


,số mẫu,tỉ lệ vỡ nợ
train,20975,0.2213
validation,8990,0.2212


## 1. PCA (#10)

Đầu vào là 33 chiều từ `build_preprocessor` (one-hot + `signed_log` cột tiền + `StandardScaler`), fit trên train. `fit_pca` giữ đủ 33 thành phần để vẽ scree.

In [3]:
pre, X_tr, X_va = rd.pca_input(df_tr, df_va)
pca = rd.fit_pca(X_tr)                      # đủ 33 thành phần
r_pca = rd.explained_ratio(pca)
assert X_tr.shape[1] == 33 and r_pca.sum() <= 1 + 1e-9 and np.allclose(r_pca, pca.explained_variance_ratio_)
rd.plot_scree(r_pca, FIG / 'pca_scree.png', 'PCA – scree và tích lũy (33 chiều)', k=K)
s_pca = rd.explained_summary(r_pca, K)
print(X_tr.shape, X_va.shape, 'tổng tỉ lệ =', round(r_pca.sum(), 6))
pd.DataFrame({'tỉ lệ': r_pca[:K], 'tích lũy': np.cumsum(r_pca)[:K]}, index=range(1, K + 1)).round(4).T

(20975, 33) (8990, 33) tổng tỉ lệ = 1.0


,1,2,3,4,5,6,7,8,9,10,11
tỉ lệ,0.2811,0.1348,0.0738,0.0613,0.0557,0.0484,0.0416,0.0355,0.0306,0.0302,0.0251
tích lũy,0.2811,0.4159,0.4896,0.5510,0.6066,0.6551,0.6966,0.7321,0.7627,0.7929,0.8180


In [4]:
pd.DataFrame([s_pca], index=['PCA']).round(4)

,n_for_80%,n_for_90%,cum_at_11
PCA,11,15,0.818


![](../results/figures/pca_scree.png)

**Nhận xét.** Cần 11 thành phần để đạt 80% và 15 để đạt 90% phương sai; 11 thành phần giữ 81,8%. Thành phần đầu giải thích 28,1% và đường tích lũy tăng chậm dần, tức thông tin không dồn vào vài trục.

### 1.2. Scatter từng cặp 5 thành phần đầu

Vẽ 3000 điểm ngẫu nhiên (seed 0) trong train, màu theo nhãn; đường chéo là mật độ từng lớp trên toàn bộ train.

In [5]:
Z_tr_full = pd.DataFrame(pca.transform(X_tr), index=X_tr.index, columns=[f'PC{i+1}' for i in range(33)])
Z_va_full = pd.DataFrame(pca.transform(X_va), index=X_va.index, columns=Z_tr_full.columns)  # val: chỉ transform
Z_pca_tr, Z_pca_va = Z_tr_full.iloc[:, :K], Z_va_full.iloc[:, :K]
pca11 = rd.fit_pca(X_tr, K)
assert np.allclose(pca11.explained_variance_ratio_, r_pca[:K]) and np.allclose(np.abs(pca11.transform(X_va)), np.abs(Z_pca_va))
_ = rd.plot_pairwise(Z_pca_tr, y_tr, 5, FIG / 'pca_pairwise.png')

![](../results/figures/pca_pairwise.png)

**Nhận xét.** Lớp vỡ nợ (đỏ) lệch về phía giá trị cao của thành phần 2 (histogram ở ô giữa hàng 2), còn ở thành phần 3–5 hai lớp gần trùng nhau. Không có cặp nào tách hai lớp thành cụm riêng, chỉ thấy độ chồng lấn giảm ở góc PC1 cao – PC2 cao.

### 1.3. Loadings 3 thành phần đầu

In [6]:
rd.plot_loadings(pca, list(X_tr.columns), 3, FIG / 'pca_loadings.png')
L_pca = rd.get_loadings(pca, X_tr.columns)
rd.top_loadings(L_pca, 3, 6)

,thành phần,hạng,đặc trưng,khối,loading
0,PC1,1,BILL_AMT4,BILL_AMT,0.840
1,PC1,2,BILL_AMT3,BILL_AMT,0.834
2,PC1,3,BILL_AMT5,BILL_AMT,0.822
3,PC1,4,BILL_AMT2,BILL_AMT,0.806
4,PC1,5,BILL_AMT6,BILL_AMT,0.775
5,PC1,6,PAY_RATIO,Dẫn xuất,-0.768
6,PC2,1,N_LATE,Dẫn xuất,0.752
7,PC2,2,PAY_1,PAY,0.642
8,PC2,3,PAY_2,PAY,0.581
9,PC2,4,PAY_3,PAY,0.530


In [7]:
rd.block_strength(L_pca, 3).round(3)

,PC1,PC2,PC3
PAY,0.294,0.332,0.002
BILL_AMT,0.418,0.065,0.000
PAY_AMT,0.153,0.348,0.000
Hồ sơ,0.011,0.079,0.994
Dẫn xuất,0.124,0.175,0.003


![](../results/figures/pca_loadings.png)

**Nhận xét.** PC1 chủ yếu là khối BILL_AMT (42% tổng loading bình phương; BILL_AMT2..5 cùng dấu, loading 0,81–0,84) cùng PAY_RATIO ngược dấu; PC2 là khối trạng thái trả, gồm PAY (33%), PAY_AMT (35%) và N_LATE (loading 0,75), nghĩa là đang trễ nhiều tháng và trả ít. PC3 gần như chỉ phản ánh hồ sơ (99,4%), nổi bật là hai cột one-hot MARRIAGE_1/MARRIAGE_2 loading ngược nhau (±0,92) và AGE.

### 1.4. Quan hệ thành phần – nhãn (validation)

Dùng validation (không tham gia fit). `auc` là ROC-AUC của chính thành phần làm điểm số, `auc_abs` bỏ qua dấu.

In [8]:
t_pca = rd.component_vs_label(Z_pca_va, y_va, 5)
rd.plot_component_boxplots(Z_pca_va, y_va, 5, FIG / 'pca_label_boxplot.png')
rd.plot_default_rate_by_component(Z_pca_va, y_va, 5, FIG / 'pca_label_default_rate.png')
t_pca.round(3)

,mean_lớp0,mean_lớp1,cohen_d,auc,auc_abs
thành phần,,,,,
1,-0.179,0.391,0.186,0.589,0.589
2,-0.395,1.436,0.924,0.737,0.737
3,0.051,-0.109,-0.103,0.468,0.532
4,0.044,-0.019,-0.044,0.464,0.536
5,0.039,-0.171,-0.155,0.471,0.529


In [9]:
pd.concat({f'PC{k}': rd.default_rate_by_quantile(Z_pca_va.iloc[:, k - 1], y_va)['default_rate'] for k in (1, 2, 3)}, axis=1).round(3).T

khoảng,1,2,3,4,5,6,7,8,9,10
PC1,0.246,0.194,0.157,0.157,0.165,0.152,0.187,0.217,0.238,0.501
PC2,0.082,0.076,0.099,0.120,0.159,0.192,0.218,0.286,0.399,0.581
PC3,0.261,0.259,0.225,0.215,0.221,0.222,0.201,0.226,0.190,0.191


![](../results/figures/pca_label_boxplot.png)

![](../results/figures/pca_label_default_rate.png)

**Nhận xét.** PC2 tách nhãn rõ nhất trên validation: AUC 0,737, Cohen's d = 0,92, và tỉ lệ vỡ nợ tăng gần đơn điệu từ 8,2% (khoảng thấp nhất) lên 58,1% (khoảng cao nhất, so với 22,1% chung). PC1 chỉ có AUC 0,589 và quan hệ không đơn điệu (50,1% ở khoảng cao nhất nhưng 24,6% ở khoảng thấp nhất); PC3–PC5 gần như không tách (AUC_abs 0,529–0,536).

## 2. FAMD (#11)

`prince.FAMD` trên dữ liệu hỗn hợp **chưa one-hot**: `SEX`, `EDUCATION`, `MARRIAGE` là category (chuỗi), 24 cột số (20 gốc + 4 dẫn xuất). Cột tiền qua `signed_log` giống đầu vào PCA để so sánh công bằng; prince tự chuẩn hóa cột số và mã hóa category kiểu MCA. Fit trên train, validation chỉ `transform`: `rd.famd_transform` mã hóa dữ liệu mới bằng thống kê train (trung bình/độ lệch chuẩn cột số, tần suất category) rồi chiếu lên SVD đã fit, vì `transform` của prince 0.23.1 tính lại tần suất category từ chính dữ liệu đưa vào.

In [10]:
famd = rd.fit_famd(df_tr, n_components=30)
r_famd = rd.explained_ratio(famd)
assert r_famd.sum() <= 1 + 1e-9 and np.isclose(famd.total_inertia_, 30) and np.isclose(r_famd.sum(), 1)
rd.plot_scree(r_famd, FIG / 'famd_scree.png', 'FAMD – scree và tích lũy quán tính (30 thành phần)', k=K)
s_famd = rd.explained_summary(r_famd, K)
Xf = rd.famd_input(df_tr)
print('cột số:', Xf.select_dtypes('number').shape[1], '| cột category:', Xf.shape[1] - Xf.select_dtypes('number').shape[1], '| tổng quán tính =', round(famd.total_inertia_, 6))
pd.DataFrame({'tỉ lệ': r_famd[:K], 'tích lũy': np.cumsum(r_famd)[:K]}, index=range(1, K + 1)).round(4).T

cột số: 24 | cột category: 3 | tổng quán tính = 30.0


,1,2,3,4,5,6,7,8,9,10,11
tỉ lệ,0.3083,0.1474,0.0597,0.0533,0.0443,0.0359,0.0351,0.0334,0.0322,0.0308,0.0251
tích lũy,0.3083,0.4557,0.5154,0.5687,0.6130,0.6489,0.6840,0.7174,0.7496,0.7805,0.8056


In [11]:
pd.DataFrame([s_famd], index=['FAMD']).round(4)

,n_for_80%,n_for_90%,cum_at_11
FAMD,11,16,0.8056


![](../results/figures/famd_scree.png)

**Nhận xét.** FAMD cần 11 thành phần để đạt 80% và 16 để đạt 90% quán tính; 11 thành phần giữ 80,6% (PCA: 81,8%). Thành phần đầu giải thích 30,8%, cao hơn PCA (28,1%).

### 2.2. Scatter từng cặp 5 thành phần đầu

In [12]:
Z_famd_tr = rd.famd_transform(famd, df_tr).iloc[:, :K]
Z_famd_va = rd.famd_transform(famd, df_va).iloc[:, :K]      # val: chỉ transform
Z_famd_tr.columns = Z_famd_va.columns = [f'F{i+1}' for i in range(K)]
assert np.allclose(Z_famd_tr.to_numpy(), famd.row_coordinates(rd.famd_input(df_tr)).iloc[:, :K].to_numpy())
_ = rd.plot_pairwise(Z_famd_tr, y_tr, 5, FIG / 'famd_pairwise.png')

![](../results/figures/famd_pairwise.png)

**Nhận xét.** Giống PCA, lớp vỡ nợ lệch về phía cao của thành phần 2, còn thành phần 3–5 hai lớp chồng nhau gần hoàn toàn. Dấu của thành phần là tùy ý nên F1 đảo dấu so với PC1; ở F1 thấp (nhóm BILL_AMT lớn) điểm đỏ dày hơn.

### 2.3. Đóng góp / tọa độ cột cho 3 thành phần đầu

Với cột số, `column_coordinates_` là tương quan với thành phần; với category là tọa độ từng mức (thang khác nên chỉ so sánh tương đối). `variable_contributions_` (mỗi biến một dòng, mỗi cột cộng lại bằng 1) cho đóng góp theo biến, cộng theo khối ở bảng dưới.

In [13]:
rd.plot_loadings(famd, None, 3, FIG / 'famd_loadings.png')
L_famd = rd.get_loadings(famd)
rd.top_loadings(L_famd, 3, 6)

,thành phần,hạng,đặc trưng,khối,loading
0,F1,1,BILL_AMT4,BILL_AMT,-0.842
1,F1,2,BILL_AMT3,BILL_AMT,-0.836
2,F1,3,BILL_AMT5,BILL_AMT,-0.824
3,F1,4,BILL_AMT2,BILL_AMT,-0.807
4,F1,5,BILL_AMT6,BILL_AMT,-0.777
5,F1,6,PAY_RATIO,Dẫn xuất,0.766
6,F2,1,N_LATE,Dẫn xuất,0.760
7,F2,2,PAY_1,PAY,0.647
8,F2,3,PAY_2,PAY,0.587
9,F2,4,PAY_3,PAY,0.537


In [14]:
vc = famd.variable_contributions_.iloc[:, :3].copy()
vc.columns = ['F1', 'F2', 'F3']
vc['khối'] = [rd.feature_block(n) for n in vc.index]
vc.groupby('khối')[['F1', 'F2', 'F3']].sum().reindex(['PAY', 'BILL_AMT', 'PAY_AMT', 'Hồ sơ', 'Dẫn xuất']).round(3)

,F1,F2,F3
khối,,,
PAY,0.293,0.343,0.289
BILL_AMT,0.420,0.063,0.119
PAY_AMT,0.155,0.348,0.008
Hồ sơ,0.008,0.067,0.066
Dẫn xuất,0.124,0.179,0.519


In [15]:
vc[['F1', 'F2', 'F3']].round(3).sort_values('F1', ascending=False).head(8)

,F1,F2,F3
variable,,,
BILL_AMT4,0.077,0.013,0.001
BILL_AMT3,0.076,0.010,0.005
BILL_AMT5,0.073,0.015,0.012
BILL_AMT2,0.070,0.005,0.039
BILL_AMT6,0.065,0.017,0.009
PAY_RATIO,0.063,0.015,0.051
PAY_5,0.060,0.034,0.090
BILL_AMT1,0.059,0.003,0.052


![](../results/figures/famd_loadings.png)

**Nhận xét.** F1 và F2 gần giống PC1 và PC2 (F1 là khối BILL_AMT 42%, F2 là trạng thái trả: PAY 34%, PAY_AMT 35%, N_LATE 0,76). Khác với PCA, F3 là xu hướng trễ hạn (LATE_TREND, loading −0,77; khối dẫn xuất 52%) còn khối hồ sơ chỉ đóng góp 6,6% ở F3, và không còn thành phần riêng cho MARRIAGE (có thể do FAMD coi SEX/EDUCATION/MARRIAGE là 3 biến category với tổng quán tính 6, thay vì 9 cột one-hot chuẩn hóa như ở PCA).

### 2.4. Quan hệ thành phần – nhãn (validation)

In [16]:
t_famd = rd.component_vs_label(Z_famd_va, y_va, 5)
rd.plot_component_boxplots(Z_famd_va, y_va, 5, FIG / 'famd_label_boxplot.png')
rd.plot_default_rate_by_component(Z_famd_va, y_va, 5, FIG / 'famd_label_default_rate.png')
t_famd.round(3)

,mean_lớp0,mean_lớp1,cohen_d,auc,auc_abs
thành phần,,,,,
1,0.177,-0.381,-0.182,0.413,0.587
2,-0.398,1.437,0.929,0.737,0.737
3,-0.013,0.089,0.076,0.504,0.504
4,-0.056,0.142,0.156,0.542,0.542
5,0.065,-0.230,-0.258,0.436,0.564


In [17]:
pd.concat({f'F{k}': rd.default_rate_by_quantile(Z_famd_va.iloc[:, k - 1], y_va)['default_rate'] for k in (1, 2, 3)}, axis=1).round(3).T

khoảng,1,2,3,4,5,6,7,8,9,10
F1,0.502,0.224,0.220,0.190,0.155,0.166,0.156,0.159,0.195,0.247
F2,0.081,0.078,0.091,0.129,0.149,0.195,0.216,0.287,0.399,0.587
F3,0.352,0.256,0.168,0.184,0.145,0.135,0.166,0.191,0.274,0.344


![](../results/figures/famd_label_boxplot.png)

![](../results/figures/famd_label_default_rate.png)

**Nhận xét.** F2 vẫn là thành phần tách nhãn tốt nhất, AUC 0,737 và tỉ lệ vỡ nợ từ 8,1% lên 58,7% theo phân vị, gần như trùng PC2. F1 có AUC_abs 0,587 (dấu đảo so với PC1) với tỉ lệ 50,2% ở khoảng thấp nhất; F3–F5 yếu (AUC_abs 0,504–0,564), nên với tách lớp FAMD và PCA khá tương đương.

## 3. Con số chuyển cho issue #12

Chỉ liệt kê số đo; bảng so sánh PCA–FAMD, t-SNE do #12 làm.

In [18]:
pd.DataFrame({'PCA': s_pca, 'FAMD': s_famd}).T.astype({'n_for_80%': int, 'n_for_90%': int}).round(4)

,n_for_80%,n_for_90%,cum_at_11
PCA,11,15,0.8180
FAMD,11,16,0.8056


**Nhận xét.** PCA và FAMD gần nhau về số thành phần cần cho 80% (đều 11) và tỉ lệ giữ tại 11 chiều (81,8% so với 80,6%); FAMD cần thêm 1 thành phần để đạt 90%. Việc so sánh khả năng giải thích và khả năng tách lớp ở dạng bảng chung để issue #12 xử lý.